In [1]:
import pdfplumber
import pandas as pd
import sys
import os
import re

In [2]:
#root = _project_root()
# --- Rutas de entrada/salida ---
rutas = {
#    "pdf_remittance": os.path.join( "..", "..", "Archivos", "Remittance", "Ecuador", "Remittance TIA.pdf"),
    "pdf_remittance": os.path.join( "..", "..", "Archivos", "Remittance", "Ecuador", "Remittance TIA 2.pdf"),
#    "fbl5n": os.path.join("..", "..", "Archivos", "Cartera", "FBL5N_Cenco.xlsx"),
    "salida": os.path.join("..", "..", "Archivos", "Template", "Colombia", "Template_HRC_Cenco.xlsx")
}
# Colocar el Customer ID del cliente
customer_id = 10267301

In [3]:
archivo_pdf = rutas["pdf_remittance"]

In [6]:
def extraer_facturas(archivo_pdf):
    filas = []

    with pdfplumber.open(rutas["pdf_remittance"]) as pdf:
        for pagina in pdf.pages:
            words = pagina.extract_words(x_tolerance=2, y_tolerance=3)
            lineas_dict = {}

            for w in words:
                y = round(w["top"])
                x = w.get("x", w.get("x0", 0))
                lineas_dict.setdefault(y, []).append((x, w["text"]))

            for y, contenido in sorted(lineas_dict.items()):
                linea = " ".join(t for _, t in sorted(contenido))
                if re.match(r"^\d{4}-\d{2}-\d{2}\s+\d+", linea):
                    filas.append(linea)

    registros = []
    for f in filas:
        numeros = re.findall(r"-?[\d,]+\.\d{2}", f)
        if len(numeros) < 3:
            continue

        bruto, ret, neto = [n.replace(",", "") for n in numeros[-3:]]
        parte_antes = f[:f.rfind(bruto)].strip()
        tokens = parte_antes.split()

        # Fecha y código plaza
        fecha = tokens[0]
        plaza_codigo = tokens[1]

        # Buscar posición del F, C o D
        tipo_idx = None
        for i, tok in enumerate(tokens):
            if tok in ["F", "C", "D"]:
                tipo_idx = i
                break

        if tipo_idx is None:
            print("⚠️ No se encontró tipo en:", f)
            continue

        # Plaza completa
        plaza_pago = " ".join(tokens[2:tipo_idx])
        tipo_letra = tokens[tipo_idx]
        documento = tokens[tipo_idx + 1]
        descripcion = " ".join(tokens[tipo_idx + 2:])

        # 🔧 Documento = Tipo + número
        documento_final = f"{tipo_letra} {documento}"
        tipo_texto = descripcion.strip()

        registros.append([
            fecha,
            f"{plaza_codigo} {plaza_pago}".strip(),
            tipo_texto,
            documento_final,
            bruto,
            ret,
            neto
        ])

    df = pd.DataFrame(
        registros,
        columns=[
            "Fecha",
            "Plaza Pago",
            "Documento",
            "Tipo",
            "Bruto",
            "Retención",
            "Neto a Pagar",
        ],
    )

    # Parche
    df[["Documento", "Tipo"]] = df[["Tipo", "Documento"]].copy()

    for c in ["Bruto", "Retención", "Neto a Pagar"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# --- Ejecutar ---
df = extraer_facturas(rutas["pdf_remittance"])

In [7]:
df.head(100)

,Fecha,Plaza Pago,Documento,Tipo,Bruto,Retención,Neto a Pagar
0,2024-05-13,100 CENTRO DE DISTRIBUCION NACIONAL,C 001014000018522,FOLIO 21013 REINTEGRO MES MAYO 68.40 0.00,68.40,0.00,68.40
1,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303974,MERCADERIA REVENTA SECCION 016,464.53,-6.06,458.47
2,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303975,"MERCADERIA REVENTA SECCION 016 2,058.43 -26.85...",2058.43,-26.85,2031.58
3,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303976,"MERCADERIA REVENTA SECCION 016 3,207.13 -41.83...",3207.13,-41.83,3165.30
4,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303977,"MERCADERIA REVENTA SECCION 014 2,604.34 -33.97...",2604.34,-33.97,2570.37
5,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303978,"MERCADERIA REVENTA SECCION 016 12,662.21 -165....",12662.21,-165.16,12497.05
6,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303979,"MERCADERIA REVENTA SECCION 016 9,001.03 -117.4...",9001.03,-117.40,8883.63
7,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303980,"MERCADERIA REVENTA SECCION 016 3,788.50 -49.42...",3788.50,-49.42,3739.08
8,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303981,"MERCADERIA REVENTA SECCION 016 5,773.59 -75.31...",5773.59,-75.31,5698.28
9,2024-05-07,100 CENTRO DE DISTRIBUCION NACIONAL,F 013002000303982,"MERCADERIA REVENTA SECCION 014 10,535.67 -137....",10535.67,-137.42,10398.25


In [15]:
print(f"\nTotal de filas: {len(df)}")


Total de filas: 50
